In [1]:
import numpy as np

In [2]:
import numpy as np
np.random.seed(42)
# ── Toy EN → FR corpus ──────────────────────────────────────────
raw_pairs = [
    (["I",   "love",  "you"],          ["je",   "t'aime"]),
    (["I",   "am",    "happy"],        ["je",   "suis",  "heureux"]),
    (["she", "is",    "kind"],         ["elle", "est",   "gentille"]),
    (["we",  "learn", "together"],     ["nous", "apprenons", "ensemble"]),
    (["he",  "reads", "books"],        ["il",   "lit",   "des", "livres"]),
    (["I",   "speak", "French"],       ["je",   "parle", "francais"]),
    (["you", "are",   "welcome"],      ["vous", "etes",  "bienvenus"]),
    (["she", "loves", "music"],        ["elle", "aime",  "la", "musique"]),
]

In [3]:
raw_pairs

[(['I', 'love', 'you'], ['je', "t'aime"]),
 (['I', 'am', 'happy'], ['je', 'suis', 'heureux']),
 (['she', 'is', 'kind'], ['elle', 'est', 'gentille']),
 (['we', 'learn', 'together'], ['nous', 'apprenons', 'ensemble']),
 (['he', 'reads', 'books'], ['il', 'lit', 'des', 'livres']),
 (['I', 'speak', 'French'], ['je', 'parle', 'francais']),
 (['you', 'are', 'welcome'], ['vous', 'etes', 'bienvenus']),
 (['she', 'loves', 'music'], ['elle', 'aime', 'la', 'musique'])]

In [4]:
# vocabulary

def build_vocab(pairs, side, specials):
    words = set()
    for pair in pairs:
        for w in pair[side]:
            words.add(w)
    vocab = specials + sorted(words)
    return {w: i for i,w in enumerate(vocab)}, {i:w for i,w in enumerate(vocab)}

In [5]:
## decode vocabulary code
words = set() # -> get unique words

for pair in raw_pairs:
    for w in pair[0]: # -> 0 is English side and 1 is French side
        words.add(w)

specials_eng = ["<PAD>", "<EOS>"]
specials_fr = ["<PAD>", "<SOS>", "<EOS>"]
vocab = specials_eng + sorted(words)
print("Unique English Words:",{w:i for i,w in enumerate(vocab)}, {i:w for i,w in enumerate(vocab)})

Unique English Words: {'<PAD>': 0, '<EOS>': 1, 'French': 2, 'I': 3, 'am': 4, 'are': 5, 'books': 6, 'happy': 7, 'he': 8, 'is': 9, 'kind': 10, 'learn': 11, 'love': 12, 'loves': 13, 'music': 14, 'reads': 15, 'she': 16, 'speak': 17, 'together': 18, 'we': 19, 'welcome': 20, 'you': 21} {0: '<PAD>', 1: '<EOS>', 2: 'French', 3: 'I', 4: 'am', 5: 'are', 6: 'books', 7: 'happy', 8: 'he', 9: 'is', 10: 'kind', 11: 'learn', 12: 'love', 13: 'loves', 14: 'music', 15: 'reads', 16: 'she', 17: 'speak', 18: 'together', 19: 'we', 20: 'welcome', 21: 'you'}


In [6]:
# make them together

en_to_ix, ix_to_en = build_vocab(raw_pairs, 0, ["<PAD>", "<EOS>"])
fr_to_ix, ix_to_fr = build_vocab(raw_pairs, 1, ["<PAD>", "<SOS>", "<EOS>"])

In [7]:
# ── Data preparation ─────────────────────────────────────────────

def make_pair(en_words, fr_words):
    src = [en_to_ix[w] for w in en_words] + [en_to_ix["<EOS>"]]
    tgt_in  = [fr_to_ix["<SOS>"]] + [fr_to_ix[w] for w in fr_words]
    tgt_out = [fr_to_ix[w] for w in fr_words] + [fr_to_ix["<EOS>"]]
    return src, tgt_in, tgt_out

dataset = [make_pair(en, fr) for en, fr in raw_pairs]

In [8]:
for i in range(len(dataset)):
    print(dataset[i])

([3, 12, 21, 1], [1, 15, 23], [15, 23, 2])
([3, 4, 7, 1], [1, 15, 22, 13], [15, 22, 13, 2])
([16, 9, 10, 1], [1, 7, 9, 12], [7, 9, 12, 2])
([19, 11, 18, 1], [1, 20, 4, 8], [20, 4, 8, 2])
([8, 15, 6, 1], [1, 14, 17, 6, 18], [14, 17, 6, 18, 2])
([3, 17, 2, 1], [1, 15, 21, 11], [15, 21, 11, 2])
([21, 5, 20, 1], [1, 24, 10, 5], [24, 10, 5, 2])
([16, 13, 14, 1], [1, 7, 3, 16, 19], [7, 3, 16, 19, 2])


In [9]:
# decode data preparation code
#en_to_ix

for en, fr in raw_pairs:
    src = [en_to_ix[w] for w in en] + [en_to_ix["<EOS>"]]
    tgt_in = [fr_to_ix["<SOS>"]] + [fr_to_ix[w] for w in fr]
    tgt_out = [fr_to_ix[w] for w in fr] + [fr_to_ix["<EOS>"]]

print(src)
print(tgt_in)
print(tgt_out)

print("src_ix:", src)   # already converted to indices!
print("words :", [ix_to_en[i] for i in src])

[16, 13, 14, 1]
[1, 7, 3, 16, 19]
[7, 3, 16, 19, 2]
src_ix: [16, 13, 14, 1]
words : ['she', 'loves', 'music', '<EOS>']


In [10]:
# lets take dataset in zero indice

src, tgt_in, tgt_out = dataset[0]

print(src)
print(tgt_in)
print(tgt_out)

print("src_ix:", src)   # already converted to indices!
print("words :", [ix_to_en[i] for i in src])

[3, 12, 21, 1]
[1, 15, 23]
[15, 23, 2]
src_ix: [3, 12, 21, 1]
words : ['I', 'love', 'you', '<EOS>']


In [11]:
# -- Hyperparameters ----------------------------------------------

D_MODEL = 16 # Embedding Dimension
D_K = 8 # query/key dimension per head
D_FF = 32 # feed forwawrd inner dimension
N_HEADS = 2 # Number of attention heads
LR = 0.01 
EPOCHS = 300

In [12]:
# Now we will define the input weigths, embeddings ....
EN_SIZE = len(en_to_ix)
FR_SIZE = len(fr_to_ix)

src_vocab_size = EN_SIZE
tgt_vocab_size = FR_SIZE

d_model = D_MODEL
d_k = D_K
d_ff = D_FF
n_heads = N_HEADS

# Embeddings
E_src = np.random.randn(src_vocab_size, d_model) * 0.01 # <- English word embeddings
E_tgt = np.random.randn(tgt_vocab_size, d_model) * 0.01 # <- French word embbeddings

print(E_src.shape)
print(E_tgt.shape)
# Note they are the dimensions, if you are interested to look in, discard .shape

(22, 16)
(25, 16)


In [13]:
# Encoder SELF-ATTENTION

W_Q_enc = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)] # <- query / key dimension per head, since there is a loop s
W_K_enc = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_V_enc = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_O_enc = np.random.randn(n_heads * d_k, d_model) * 0.01

# What is W_O_enc?
# W_O_enc is the output projection matrix of the multi-head attention in the encoder. It comes after all the attention heads are concatenated together.

print(W_Q_enc[0].shape)
print(W_K_enc[0].shape)
print(W_V_enc[0].shape)
print(W_O_enc.shape)

(16, 8)
(16, 8)
(16, 8)
(16, 16)


In [14]:
## Encoder Feed Forward Network + 2 Biases , take care on matrix dimensions and their orders, because they are essential for matrix multiplications
W1_enc = np.random.randn(d_model,d_ff) * 0.01
b1_enc = np.zeros((d_ff))
W2_enc = np.random.randn(d_ff, d_model) * 0.01
b2_enc = np.zeros((d_model))

print(W1_enc.shape)
print(b1_enc.shape)
print(W2_enc.shape)
print(b2_enc.shape)

(16, 32)
(32,)
(32, 16)
(16,)


In [15]:
# Decoder masked Self-Attention
W_Q_dec = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_K_dec = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_V_dec = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_O_dec = np.random.randn(n_heads * d_k, d_model) * 0.01

print(W_Q_dec[0].shape)
print(W_K_dec[0].shape)
print(W_V_dec[0].shape)
print(W_O_dec.shape)

(16, 8)
(16, 8)
(16, 8)
(16, 16)


In [16]:
# Decoder cross-attention (Q from decoder , K/V from encoder)
W_Q_cross = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_K_cross = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_V_cross = [np.random.randn(d_model, d_k) * 0.01 for _ in range(n_heads)]
W_O_cross = np.random.randn(n_heads * d_k, d_model) * 0.01

print(W_Q_cross[0].shape)
print(W_K_cross[0].shape)
print(W_V_cross[0].shape)
print(W_O_cross.shape)

(16, 8)
(16, 8)
(16, 8)
(16, 16)


In [17]:
# Decoder FFN (2 MATRICES + 2 BIASES)
W1_dec = np.random.randn(d_model,d_ff) * 0.01
b1_dec = np.zeros((d_ff,))
W2_dec = np.random.randn(d_ff, d_model) * 0.01
b2_dec = np.zeros((d_model,))

print(W1_dec.shape)
print(b1_dec.shape)
print(W2_dec.shape)
print(b2_dec.shape)

(16, 32)
(32,)
(32, 16)
(16,)


In [18]:
# Output layer
W_out = np.random.randn(d_model, tgt_vocab_size) * 0.01
b_out = np.zeros((tgt_vocab_size,))

print(W_out.shape)
print(b_out.shape)

(16, 25)
(25,)


In [19]:
# Now Positional Encoding, it is calculation depends on sine and cosine

# positional encoding
def positional_encoding(seq_len):
    PE = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            PE[pos, i]   = np.sin(pos / 10000 ** (2*i / d_model))
            PE[pos, i+1] = np.cos(pos / 10000 ** (2*i / d_model))
    return PE

# toy example
#SRC = ["I", "am", "a", "student", "<EOS>"]
#src_ix = [0, 1, 2, 3, 4]    

src_ix = src

pe = positional_encoding(len(src_ix))
pe.shape

x = E_src[src_ix]
x = x + pe # <- take care here
x.shape 

(4, 16)

In [20]:
# softmax
def softmax(x):
    x = x - x.max(axis = -1 , keepdims = True) # numerical stability
    return np.exp(x) / np.exp(x).sum(axis = -1, keepdims = True)

In [21]:
# layer normalization
def layer_norm(x, eps = 1e-6):
    mean = x.mean(axis = -1, keepdims = True)
    std = x.std(axis = -1, keepdims = True)
    return (x - mean) / (std + eps)

In [22]:
# now multi-head attention
# multihead attention
def multihead_attention(Q_in, K_in, V_in, W_Qs, W_Ks, W_Vs, W_O, mask=None):
    heads = []
    for h in range(n_heads):
        Q = Q_in @ W_Qs[h]
        K = K_in @ W_Ks[h]
        V = V_in @ W_Vs[h]
        scores = Q @ K.T / np.sqrt(d_k) # (seq, seq)
        if mask is not None:
            scores = scores + mask
        attn = softmax(scores)
        heads.append(attn @ V)
    concat = np.concatenate(heads, axis = 1) # (seq, n_heads * d_k)
    return concat @ W_O # (seq, d_model)

In [23]:
# decode function
Q_enc  =  x
K_enc  = x
V_enc  = x
mask = None

heads = []
for h in range(n_heads):
    Q = Q_enc @ W_Q_enc[h]
    K = K_enc @ W_K_enc[h]
    V = V_enc @ W_V_enc[h]
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    attn = softmax(scores)
    heads.append(attn @ V)

concat = np.concatenate(heads, axis = 1)

print(Q.shape)
print(K.shape)
print(V.shape)
print((concat @ W_O_enc).shape)

(4, 8)
(4, 8)
(4, 8)
(4, 16)


In [24]:
attn = concat @ W_O_enc
x = layer_norm(x + attn)

print(x.shape)

(4, 16)


In [25]:
## Now we will go into encoder step

# we have already computed these steps:
# x = x + self.positional_encoding(len(src_ix))

# step 2: Multihead Self-Attention
# attn = self.multihead_attention(x, x, x, self.W_Q_enc, self.W_K_enc, self.W_V_enc, self.W_O_enc)

# step 3: Add & Norm
# x = self.layer_norm(x + attn)

# now contniue from step 4:
x_enc_before_ffn = x
enc_ffn_hidden = np.maximum(0, x @ W1_enc + b1_enc) # ReLU
ffn = enc_ffn_hidden @ W2_enc + b2_enc

x = layer_norm(x + ffn)

enc_cache = (x_enc_before_ffn, enc_ffn_hidden, src_ix)
print(x.shape)
#print(enc_cache)

(4, 16)


In [26]:
## now decoder part !!

src, tgt_in, tgt_out = dataset[0]   # first sentence pair

tgt_ix = tgt_in

y = E_tgt[tgt_ix]

print(y.shape)

y = y + positional_encoding(len(tgt_ix))

# build causal mask so position t cannot see t+1, t+2....
T = len(tgt_ix)

mask = np.triu(np.ones((T,T)), k=1) * -1e9

attn = multihead_attention(y, y, y, W_Q_dec, W_K_dec, W_V_dec, W_O_dec, mask=mask)

y = layer_norm(y + attn)

## take care here !!
Q_dec = y
K_enc = x
V_enc = x



# step 4: cross-attention — explicit loop to save attn weights & V per head
# Q comes from decoder ("what am I looking for?")
# K and V come from encoder ("what does each source word contain?")

cross_heads = []
cross_attn_ws = []
cross_Vs = []


for h in range(n_heads):
    Q_h = Q_dec @ W_Q_cross[h]
    K_h = K_enc @ W_K_cross[h]
    V_h = V_enc @ W_V_cross[h]
    sc = Q_h @ K_h.T / np.sqrt(d_k)
    aw = softmax(sc)
    cross_attn_ws.append(aw)
    cross_Vs.append(V_h)
    cross_heads.append(aw @ V_h)

cross = np.concatenate(cross_heads, axis=1) @ W_O_cross

print(cross.shape)

y = layer_norm(y + cross)

y_before_ffn = y

y_ffn_hidden = np.maximum(0, y @ W1_dec + b1_dec) # RELU
y_ffn = y_ffn_hidden @ W2_dec + b2_dec

y = layer_norm(y + y_ffn)

dec_cache = (y_before_ffn, y_ffn_hidden, cross_attn_ws, cross_Vs)

print(y.shape)

(3, 16)
(3, 16)
(3, 16)


In [27]:
# now forward method 

enc_out, enc_cache = x, enc_cache
dec_out, dec_cache = y, dec_cache

logits = dec_out @ W_out + b_out
logits -= logits.max(axis = 1, keepdims = True)
probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)

print("probs shape", probs.shape)

probs shape (3, 25)


In [28]:
# What does the model predict at each position?
predicted_ix = np.argmax(probs, axis=1)
predicted_words = [ix_to_fr[i] for i in predicted_ix]
print("Target input  :", [ix_to_fr[i] for i in tgt_in])   # what decoder read
print("Target output :", [ix_to_fr[i] for i in tgt_out])  # what it should predict
print("Model predicts:", predicted_words)                  # what it actually predicts

Target input  : ['<SOS>', 'je', "t'aime"]
Target output : ['je', "t'aime", '<EOS>']
Model predicts: ['francais', 'heureux', 'suis']


In [29]:
## now train step !

# we will use -> probs, dec_out, dec_cache, enc_cache

y_before_ffn, y_ffn_hidden, cross_attn_ws, cross_Vs = dec_cache
x_enc_before_ffn, enc_ffn_hidden, src_ix = enc_cache


# loss -> cross-entropy
tgt_out_ix = tgt_out
loss = sum(-np.log(probs[t, tgt_out_ix[t]] + 1e-9) for t in range(T)) / T

# backward pass
d_logits = probs.copy()

for t, ix in enumerate(tgt_out_ix):
    d_logits[t,ix] -= 1
d_logits /= T

In [30]:
# STEP 2: Gradients for W_out and b_out 
dW_out = dec_out.T @ d_logits
db_out = d_logits.sum(axis=0)

print(dW_out.shape)
print(db_out.shape)

(16, 25)
(25,)


In [34]:
# step 3: Gradient flowing back into dec_out
d_dec_out = d_logits @ W_out.T

# step 4 through layer norm
d_ffn = d_dec_out

# step 5: gradient through W2_dec
dW2_dec = y_ffn_hidden.T @ d_ffn
db2_dec = d_ffn.sum(axis=0)

# step 6: back through ReLU and W1_dec
d_ffn_hidden = d_ffn @ W2_dec.T
d_ffn_hidden[y_ffn_hidden <= 0] = 0 #ReLu mask
dW1_dec = y_before_ffn.T @ d_ffn_hidden
db1_dec = d_ffn_hidden.sum(axis=0)

# step 7: gradient into target embeddings
d_emb = d_ffn_hidden @ W1_dec.T
dE_tgt = np.zeros_like(E_tgt)
for t, ix in enumerate(tgt_ix):
    dE_tgt[ix] += d_emb[t]

# step 8: Gradient through cross-attention to enc_out (V path)
# d_emb is approximately the gradient w.r.t. cross_attn_output
d_concat = d_emb @ W_O_cross.T
d_enc_out = np.zeros((len(src_ix), d_model))
for h in range(n_heads):
    d_head = d_concat[:, h*d_k:(h+1)*d_k]
    d_V_h = cross_attn_ws[h].T @ d_head
    d_enc_out += d_V_h @ W_V_cross[h].T

# step 9: gradient through encoder FFN
d_enc_ffn_hidden = d_enc_out @ W2_enc.T
d_enc_ffn_hidden[enc_ffn_hidden <= 0] = 0              # ReLU mask
d_enc_x = d_enc_ffn_hidden @ W1_enc.T 

dE_src = np.zeros_like(E_src)
for t,ix in enumerate(src_ix):
    dE_src[ix] += d_enc_x[t]

lr = LR

# Update Weights
W_out  -= lr * dW_out
b_out  -= lr * db_out
W2_dec -= lr * dW2_dec
b2_dec -= lr * db2_dec
W1_dec -= lr * dW1_dec
b1_dec -= lr * db1_dec
E_tgt  -= lr * dE_tgt
E_src  -= lr * dE_src     

#### by Hincal Topcuoglu